# POC véhicules intermédiaires : réparation, marché et emploi

Ce notebook explore les données ouvertes de l'ADEME (registre SYDEREP) et de France Travail.

**Mode d'emploi** : cliquez sur *Exécution > Tout exécuter*. Chaque cellule affiche un résultat. Rien n'est à installer.

## 1. Préparation

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt

API = "https://data.ademe.fr/data-fair/api/v1/datasets/{id}/lines"

def lire_ademe(dataset_id, **filtres):
    """Lit toutes les lignes d'un jeu de données ADEME."""
    r = requests.get(API.format(id=dataset_id), params={**filtres, "size": 10000}, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["results"])

## 2. Réparations de vélos par département

Source : ADEME, *REP - Réparation des produits usagés*, filière ASL (articles de sport et loisirs), catégorie 1 (cycles, à confirmer).

In [ ]:
rep = lire_ademe("kzuew6wsvf2efs9xo-hf-pkl", filiere_eq="ASL", categorie_eq="CAT_1",
                 select="annee,dep_reparation,nombre_reparations")
rep = rep.groupby(["annee", "dep_reparation"], as_index=False)["nombre_reparations"].sum()
print(rep.groupby("annee")["nombre_reparations"].sum())

In [ ]:
derniere = rep["annee"].max()
top = rep[rep["annee"] == derniere].nlargest(15, "nombre_reparations").sort_values("nombre_reparations")
top.plot.barh(x="dep_reparation", y="nombre_reparations", legend=False, figsize=(8, 6),
              title=f"Réparations de vélos aidées en {derniere} : 15 premiers départements")
plt.xlabel("Nombre de réparations"); plt.ylabel("Département"); plt.show()

## 3. Véhicules de catégorie L mis sur le marché

In [ ]:
msm = lire_ademe("wowibc83bqmkehe2w8krhzdo", filiere_in="VHCL,ASL",
                 select="annee,filiere,specif1,quantite,tonnage")
msm = msm.groupby(["annee", "filiere", "specif1"], as_index=False)[["quantite", "tonnage"]].sum(min_count=1)
msm["kg_par_vehicule"] = (msm["tonnage"] * 1000 / msm["quantite"]).round(0)
msm

## 4. Producteurs inscrits dans la filière véhicules (VHCL)

In [ ]:
prod = lire_ademe("rep-producteurs-idu", Filiere_eq="VHCL", select="Raison_Sociale,Pays")
print(len(prod), "producteurs")
prod.sort_values("Raison_Sociale")

## 5. Emploi : enquête BMO 2026 (France Travail)

Changez `MOT_CLE` pour choisir les métiers (par exemple `"cycles"`, `"véhicules"` ou `"mécanicien"`). Les noms de colonnes peuvent varier d'une année à l'autre : la cellule affiche les colonnes du fichier pour vérifier.

In [ ]:
from io import BytesIO
URL_BMO = ("https://www.francetravail.org/files/live/sites/peorg/files/documents/"
           "Statistiques-et-analyses/Open-data/BMO/Base-open-data-BMO-2026.xlsx")
contenu = requests.get(URL_BMO, timeout=60).content

apercu = pd.read_excel(BytesIO(contenu), header=None, nrows=15)
entete = next((i for i, l in apercu.iterrows()
              if "met" in [str(v).strip().lower() for v in l]
              or any("métier" in str(v).lower() for v in l)), 0)
bmo = pd.read_excel(BytesIO(contenu), header=entete)
print(list(bmo.columns))

In [ ]:
MOT_CLE = "cycles"
COL_METIER, COL_DEP, COL_PROJETS, COL_DIFFICILES = "Nom métier BMO", "Dept", "met", "xmet"  # à adapter si besoin

sel = bmo[bmo[COL_METIER].astype(str).str.contains(MOT_CLE, case=False, na=False)].copy()
print("Métiers retenus :", sel[COL_METIER].unique())
for c in (COL_PROJETS, COL_DIFFICILES):
    sel[c] = pd.to_numeric(sel[c].astype(str).str.replace("*", "", regex=False), errors="coerce")
sel["dep"] = sel[COL_DEP].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(2)
emploi = sel.groupby("dep", as_index=False)[[COL_PROJETS, COL_DIFFICILES]].sum()

croise = emploi.merge(rep[rep["annee"] == derniere], left_on="dep", right_on="dep_reparation", how="outer")
croise.sort_values(COL_PROJETS, ascending=False).head(20)